# **One chain rule: checking the equivalence theorem**

A practice for the module ["Attribution from axioms"](https://ai-interpretability.school).

The lesson makes a strong and surprising claim: **Gradient×Input, $\varepsilon$-LRP and
DeepLIFT (Rescale) compute exactly the same number under three conditions.** Three methods from
three different papers, with three different motivations.

This is the only claim in the course that can be checked **to the last digit**. Not "the maps
look similar", not "the correlation is high" — literally the same number. Let us check.

Along the way we will answer a question the lesson leaves open: *why* does completeness come
for free on such a network.

It runs instantly: the network is small and needs no training.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)
DIMS = [5, 8, 6, 3]     # layer sizes: input, two hidden, output
TARGET = 1              # the class we compute the attribution for


def build(bias, act=nn.ReLU):
    """A toy network. No training needed: the point is in the propagation rules, not the weights."""
    layers = []
    for a, b in zip(DIMS, DIMS[1:]):
        layers += [nn.Linear(a, b, bias=bias), act()]
    return nn.Sequential(*layers[:-1]).eval()


def trace(net, x):
    """Activations before and after each layer — all three methods need them."""
    acts = [x]
    for layer in net:
        x = layer(x)
        acts.append(x)
    return acts


x = torch.rand(1, 5) + 0.2
zero = torch.zeros(1, 5)
print(f'input:', x.numpy().round(3))

## 1. Three methods, three dozen lines

The key to seeing the equivalence is to write all three **yourself** rather than call a library.
A library would hide the very place the whole thing is about: what each method substitutes for
the derivative of the nonlinearity.

Look at the nonlinearity in each of the three functions:

- `grad_x_input` — the derivative is computed by `backward()`; for a ReLU that is the indicator
  $[z>0]$;
- `eps_lrp` — relevance passes through the nonlinearity **untouched**; only the linear layer
  redistributes it;
- `deeplift_rescale` — the secant $\dfrac{g(z)-g(z^0)}{z-z^0}$.

Three different rules. We are about to see that on a ReLU they coincide.

In [ ]:
def grad_x_input(net, x, c):
    """Gradient x Input: the gradient with respect to the input, times the input itself."""
    x = x.clone().requires_grad_(True)
    net(x)[0, c].backward()
    return (x * x.grad).detach()


def eps_lrp(net, x, c, eps=1e-9):
    """eps-LRP: relevance flows back through the layers, the denominator carries a stabilizer."""
    acts = trace(net, x)
    R = torch.zeros_like(acts[-1])
    R[0, c] = acts[-1][0, c]                 # the relevance of the output equals the logit itself
    for i in range(len(net) - 1, -1, -1):
        layer, a_in = net[i], acts[i]
        if isinstance(layer, nn.Linear):
            z = layer(a_in)
            R = a_in * ((R / (z + eps * torch.sign(z) + (z == 0) * eps)) @ layer.weight)
        # a nonlinearity does not redistribute relevance: it passes straight through
    return R.detach()


def deeplift_rescale(net, x, baseline, c):
    """DeepLIFT (Rescale): secant multipliers, the chain rule, times the deviation of the input."""
    a_x, a_0 = trace(net, x), trace(net, baseline)
    m = torch.zeros_like(a_x[-1])
    m[0, c] = 1.0
    for i in range(len(net) - 1, -1, -1):
        layer = net[i]
        if isinstance(layer, nn.Linear):
            m = m @ layer.weight             # on a linear layer the multiplier simply goes through the weights
        else:
            dz, dx = a_x[i + 1] - a_0[i + 1], a_x[i] - a_0[i]
            safe = torch.where(dx.abs() > 1e-7, dx, torch.ones_like(dx))
            m = m * torch.where(dx.abs() > 1e-7, dz / safe, torch.zeros_like(dx))
    return (m * (x - baseline)).detach()

## 2. The conditions hold — let us check the numbers

In [ ]:
def compare(title, net, x, baseline):
    g = grad_x_input(net, x, TARGET)
    l = eps_lrp(net, x, TARGET)
    d = deeplift_rescale(net, x, baseline, TARGET)
    delta = net(x)[0, TARGET].item() - net(baseline)[0, TARGET].item()
    print(f'{title}')
    print(f'   Grad x Input against eps-LRP    max discrepancy {(g - l).abs().max():.2e}')
    print(f'   Grad x Input against DeepLIFT   max discrepancy {(g - d).abs().max():.2e}')
    print(f'   sum of attributions {g.sum():.6f}   f(x) - f(baseline) {delta:.6f}')


compare(f'The conditions of the theorem hold: ReLU, no biases, zero baseline', build(bias=False), x, zero)

**Look at the second line: the discrepancy is exactly zero.** Not "small", not
"within precision" — `0.00e+00`, a bit-for-bit match. Grad×Input and DeepLIFT with a zero
baseline on a ReLU network without biases are literally the same computation written in
different words.

For $\varepsilon$-LRP the discrepancy is of order $10^{-8}$ — that is the stabilizer
$\varepsilon$, which we set to a non-zero value so as not to divide by zero. Drive it to zero
and that goes away too.

**And the third line: the sum of the attributions is exactly $f(x) - f(x')$.** Completeness,
the property Integrated Gradients was built for, appears here by itself, in a method that never
promised it. Why — in section 4.

**Task 1.** Change `TARGET` to another class and `torch.manual_seed` to another number. Does the
bit-for-bit match survive? And what about the order of the $\varepsilon$-LRP discrepancy?

In [ ]:
# Your code here

## 3. Breaking the conditions one at a time

The lesson names three conditions: the nonlinearities are piecewise linear and pass through
zero, the baseline is zero, the network has no biases. Let us check each — break one at a time
and see what exactly diverges.

In [ ]:
compare(f'Condition 3 broken: the layers now have biases', build(bias=True), x, zero)
compare(f'Condition 1 broken: tanh instead of ReLU', build(bias=False, act=nn.Tanh), x, zero)
compare(f'Condition 2 broken: the baseline is not zero', build(bias=False), x, torch.rand(1, 5) * 0.3)

We read the result line by line, and each line says its own thing.

**Biases.** DeepLIFT diverges, while $\varepsilon$-LRP still matches Grad×Input. But the most
important part is the third line: **completeness has broken** — the sum of the attributions no
longer equals the deviation of the output, and the gap is not small. The reason is substantive:
a bias contributes to the output but **belongs to no input feature**. There is nothing left to
decompose the output over the features without a remainder.

**tanh instead of ReLU.** Both methods diverge. As expected: for a smooth function the secant
and the tangent do not coincide, and the ratio $g(z)/z$ for a tanh equals neither.

**A non-zero baseline.** DeepLIFT diverges — it is the only one of the three that uses a
baseline at all. $\varepsilon$-LRP did not notice the change of baseline because it does not
know one exists: its reference point is wired into the rule and is always zero.

**Task 2.** Break two conditions at once — biases and tanh, say. Do the discrepancies add up,
or does one mask the other? And a separate question: under which of the three violations does
the map change **substantively** rather than numerically?

In [ ]:
# Your code here

## 4. Why completeness came for free

The lesson says that on such a network the sum of Grad×Input equals the output, but not where
that comes from. Let us check a guess.

A ReLU network without biases has a special property: multiply the input by a positive number
and the output is multiplied by the same number. Such functions are called **positively
homogeneous of degree one**.

In [ ]:
net = build(bias=False)
base = net(x)[0, TARGET].item()
for k in (0.5, 1.0, 2.0, 3.0):
    print(f'   f({k}x) / f(x) = {net(k * x)[0, TARGET].item() / base:.4f}   expected {k}')

It matches to four decimals for any $k$. This is no accident: a ReLU without a bias
satisfies $\mathrm{ReLU}(kz) = k\,\mathrm{ReLU}(z)$ for $k>0$, and a linear layer without a
bias all the more so. The property passes through the whole network.

And for a homogeneous function of degree one, **Euler's homogeneous function theorem** holds:

$$\sum_i x_i \frac{\partial f}{\partial x_i} = f(x)$$

On the left is exactly the sum of the Gradient×Input attributions. On the right is the output of
the network.

**Hence an important conclusion worth taking away.** Completeness for Gradient×Input on such a
network is a **property of the network, not a property of the method**. Add biases and it
disappears, which is what we saw in section 3. The completeness of Integrated Gradients is a
different thing: it is proved for **any** differentiable model and any baseline. That is the
difference between "got lucky" and "guaranteed", and the whole module was written for it.

**Task 3.** Convince yourself: take a network **with** biases and check the homogeneity the same
way. Then compute the sum of Grad×Input for it and compare with $f(x)-f(0)$. How large is the
gap, and does it grow with the number of layers?

In [ ]:
# Your code here

## What to take away from this notebook

- **The theorem is not a figure of speech.** Under the stated conditions Grad×Input and DeepLIFT
  match bit for bit. If yours diverged, look for a bug in the code, not for a substantive
  difference. That is exactly the practical consequence the equivalence theorem gives.
- **Each condition answers for its own thing.** Biases break completeness, a smooth nonlinearity
  separates the secant from the tangent, a non-zero baseline moves only DeepLIFT. Knowing which
  condition is violated is more useful than knowing that the methods diverged.
- **Completeness sometimes comes for free, and that is dangerous.** On a ReLU network without
  biases it follows from the homogeneity of the network, not from the properties of the method.
  Carry the same method over to a network with biases and you quietly lose the guarantee without
  being told — unless you check the sum.
- **What you choose is not a method but two decisions:** the rule for the nonlinearity and the
  baseline. Everything else follows.